# A–φ, T–Ω, A–T, three dimensions, gauges, and boundary conditions

This explains **how fields are obtained** within the field-to-circuit construction introduced in [What is CLN?](00_what_is_cln.ipynb). The symbolic examples check relationships between 3D potentials and physical fields; they are not 3D FEM convergence tests.

## Unknowns and the physical fields to compare

| Formulation | Representative reconstruction | Main checks |
|---|---|---|
| A–φ | $B=\nabla\times A$, $E=-sA-\nabla\phi$ | Current continuity, terminal voltage, gauge |
| T–Ω | $J=\nabla\times T$, $H=H_s+T+\nabla\Omega$ | Flux continuity, known excitation field, circulation |
| A–T | $B=\nabla\times A$, $J=\nabla\times T$ | Coupled Ampère/Faraday residuals |

Implementations with the same formulation name may use different signs and scaling for scalar potentials. For example, if $E=-s(A+\nabla\phi_c)$, the table's $\phi=s\phi_c$. Distinguish the conductor supporting T from the exterior requiring Ω. Non-simply-connected domains also require circulation degrees of freedom.

## Gauge correction does not change physical fields

For A–φ, $A\mapsto A+\nabla\chi$, $\phi\mapsto\phi-s\chi$. For T–Ω, $T\mapsto T+\nabla\chi$, $\Omega\mapsto\Omega-\chi$. Paired transformations preserve E/B or J/H.

The following polynomial 3D potentials verify this invariance symbolically. They do not guarantee exact representation of the correction in arbitrary finite-element spaces.


In [1]:
import sympy as sp
x,y,z,s=sp.symbols('x y z s', real=True)
coords=(x,y,z)
def grad(f):return sp.Matrix([sp.diff(f,c) for c in coords])
def curl(v):return sp.Matrix([sp.diff(v[2],y)-sp.diff(v[1],z),sp.diff(v[0],z)-sp.diff(v[2],x),sp.diff(v[1],x)-sp.diff(v[0],y)])
A=sp.Matrix([x*y,y*z,z*x]);phi=x*y*z;chi=x**2*y+z**3
E=-s*A-grad(phi);B=curl(A)
assert sp.simplify(-s*(A+grad(chi))-grad(phi-s*chi)-E)==sp.zeros(3,1)
assert sp.simplify(curl(A+grad(chi))-B)==sp.zeros(3,1)
T=sp.Matrix([y*z,x*z,x*y**2]);Omega=x*z
assert sp.simplify(T+grad(chi)+grad(Omega-chi)-(T+grad(Omega)))==sp.zeros(3,1)
assert sp.simplify(curl(T+grad(chi))-curl(T))==sp.zeros(3,1)
print('PASS: A/phi preserves E and B; T/Omega preserves J and H')


PASS: A/phi preserves E and B; T/Omega preserves J and H


## Start boundary-condition comparisons from the boundary term

$$\int_\Omega(\nabla\times u)\cdot v\,dV=\int_\Omega u\cdot(\nabla\times v)\,dV+\int_{\partial\Omega}(n\times u)\cdot v\,dS.$$

This term identifies the prescribed physical quantity. Record open exterior boundaries, conductor interfaces, terminal excitation, homogeneous stage corrections, and gauge constraints separately. Comparing only the labels Dirichlet/Neumann is insufficient.

For the circular-wire Type2 case, the same normalized next-stage solution simultaneously satisfies the A formulation's maintained boundary electric field and the T formulation's zero boundary magnetic field. **The existing boundary conditions and normalization were not wrong; the explanation had not traced reconstruction of fields from the unknowns.** See the [derivation](type2_same_circuit.ipynb).

## Reading 3D data

Tanimoto's A–φ/T–Ω/A–T research notebooks contain cylindrical 3D meshes, HCurl/H1 spaces, side/end constraints, and gradient-component treatments. They provide the starting point for 3D reproduction. The original notebooks and their full runtime environment are not bundled in this release.

| Check | What it establishes |
|---|---|
| Circular wire on a 3D mesh | Consistency of implemented spaces, boundaries, and assembly in a symmetric case |
| Axially varying excitation or geometry | Nontrivial field reconstruction including φ/Ω |
| Changed gauge treatment or penalty coefficient | Gauge-dependent error in physical fields and circuit constants |
| Changed mesh or order | Separation of discretization error from formulation differences |

Compare reconstructed E/B/J/H at matched coordinates, energies, terminal quantities, and residuals, rather than potential coefficients. Even when a positive-shift mass term suppresses the nullspace of an isolated T space, gauges in all mixed spaces are not necessarily resolved automatically.

This notebook verifies the gauge identities above, not general numerical validation of 3D CLN.


## Reproduce actual 3D FEM calculations

The [3D circular-wire reproduction](06_3d_round_wire.ipynb) uses three formulations based on student notebooks. It retains their boundary conditions, makes gauges and normalization explicit, and compares circuit constants and physical fields against analytic expressions. Scope: linear material, s₀=0, conductor interior, initial stages.
